# K-Nearest Neighbors and Support Vector Machines

Two fundamentally different classification algorithms: KNN (instance-based, non-parametric) and SVM (margin-based, with kernel trick).

## K-Nearest Neighbors (KNN)

**How it works**: Given a new data point, find the *K* closest training samples and vote on the class (majority vote for classification, average for regression).

### Distance Metrics
- **Euclidean** (L2): $\sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$
- **Manhattan** (L1): $\sum_{i=1}^{n} |x_i - y_i|$
- **Minkowski** (generalized): $(\sum_{i=1}^{n} |x_i - y_i|^p)^{1/p}$

### Choosing K
- Small K (e.g., 1): low bias, high variance (overfits)
- Large K: higher bias, lower variance (smoother boundary)
- Typically choose odd K for binary classification to avoid ties
- $K \approx \sqrt{n}$ is a common heuristic

KNN is a **lazy learner** — no training phase; all computation happens at prediction time.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# Load Digits dataset (8x8 images of digits 0-9)
digits = load_digits()
X, y = digits.data, digits.target

print(f"Digits dataset: {X.shape[0]} samples, {X.shape[1]} features (8x8 pixels)")
print(f"Classes: {digits.target_names}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Display sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Digit: {digits.target[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Split and scale (KNN needs scaling!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# KNN with default K=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred_knn = knn.predict(X_test_s)

print("KNN (K=5) on Digits:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

In [ ]:
# Finding the optimal K
k_range = range(1, 31, 2)
cv_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_temp, X_train_s, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]
best_score = max(cv_scores)

plt.figure(figsize=(8, 4))
plt.plot(k_range, cv_scores, 'bo-', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.7)
plt.xlabel('K (number of neighbors)')
plt.ylabel('Cross-validated accuracy')
plt.title(f'KNN: Optimal K = {best_k} (CV accuracy = {best_score:.4f})')
plt.grid(alpha=0.3)
plt.show()
print(f"Best K value: {best_k} with CV accuracy: {best_score:.4f}")

In [ ]:
# Compare distance metrics
metrics = ['euclidean', 'manhattan', 'minkowski']
for metric in metrics:
    knn_m = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    knn_m.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, knn_m.predict(X_test_s))
    print(f"  Metric '{metric:12s}' accuracy: {acc:.4f}")

## Support Vector Machines (SVM)

SVM finds the **hyperplane** that maximizes the margin between classes.

### Key Concepts
- **Support Vectors**: data points closest to the decision boundary (they "support" the margin)
- **Margin**: distance between the hyperplane and the closest support vectors
- **Hard Margin**: perfectly separates classes (only possible if data is linearly separable)
- **Soft Margin**: allows some misclassifications (controlled by parameter C)

### The Kernel Trick
SVM can project data into higher dimensions using a **kernel function**, enabling non-linear decision boundaries:
- **Linear Kernel**: $K(x, y) = x \cdot y$
- **Polynomial Kernel**: $K(x, y) = (\gamma x \cdot y + r)^d$
- **RBF (Radial Basis Function)**: $K(x, y) = \exp(-\gamma ||x - y||^2)$

### Key Parameters
- **C**: regularization parameter (small C = wider margin, may misclassify more; large C = narrower margin, fewer training errors)
- **$\gamma$** (gamma): influence of a single training point (small = far influence; large = close influence only)

In [ ]:
# SVM with different kernels
kernels = ['linear', 'poly', 'rbf']
svm_results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, gamma='scale', random_state=42)
    svm.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_s))
    svm_results[kernel] = acc
    print(f"SVM ({kernel:8s}) accuracy: {acc:.4f}")

In [ ]:
# Tuning SVM: C and gamma for RBF kernel
C_values = [0.1, 1, 10, 100]
gamma_values = [0.001, 0.01, 0.1, 1]

for C in C_values:
    for gamma in gamma_values:
        svm_temp = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        svm_temp.fit(X_train_s, y_train)
        acc = accuracy_score(y_test, svm_temp.predict(X_test_s))
        if acc > 0.97:
            print(f"  C={C:5.1f}, gamma={gamma:.3f} -> accuracy = {acc:.4f}")

In [ ]:
# Visualizing SVM decision boundaries (using 2D PCA projection)
from sklearn.decomposition import PCA

# Reduce digits to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_s)

# Train SVM on 2D data
svm_2d = SVC(kernel='rbf', C=10, gamma=0.1, random_state=42)
svm_2d.fit(X_pca, y_train)

# Create mesh
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, 
                      cmap='tab10', edgecolors='k', alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('SVM (RBF) Decision Regions on Digits (2D PCA)')
plt.colorbar(scatter)
plt.show()

In [ ]:
# When to use KNN vs SVM?
print("""
USE KNN when:
- You want a simple, interpretable baseline
- Decision boundaries are irregular
- Dataset is small to medium (KNN is expensive at prediction time)
- You don't need probabilistic outputs (though KNN can provide them)

USE SVM when:
- You have clear margin separation between classes
- You need to model non-linear boundaries (RBF kernel)
- You have medium-sized datasets with many features
- You need good generalization with limited data

BOTH require feature scaling!
""")

In [ ]:
# Problem: Breast Cancer classification with KNN and SVM
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

scaler_c = StandardScaler()
X_c_train_s = scaler_c.fit_transform(X_c_train)
X_c_test_s = scaler_c.transform(X_c_test)

# KNN
knn_c = KNeighborsClassifier(n_neighbors=5)
knn_c.fit(X_c_train_s, y_c_train)
knn_acc = accuracy_score(y_c_test, knn_c.predict(X_c_test_s))

# SVM (RBF)
svm_c = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm_c.fit(X_c_train_s, y_c_train)
svm_acc = accuracy_score(y_c_test, svm_c.predict(X_c_test_s))

print("Breast Cancer Classification:")
print(f"  KNN (K=5):  {knn_acc:.4f}")
print(f"  SVM (RBF):  {svm_acc:.4f}")

In [ ]:
# Problem: SVM with different kernels on a non-linear dataset
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=200, noise=0.15, random_state=42)
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    svm_m = SVC(kernel=kernel, C=10, gamma=1, random_state=42)
    svm_m.fit(X_m_train, y_m_train)
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-2, 3, 100), np.linspace(-1.5, 2, 100))
    Z = svm_m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_m_train[:, 0], X_m_train[:, 1], c=y_m_train, cmap='coolwarm', edgecolors='k')
    acc = accuracy_score(y_m_test, svm_m.predict(X_m_test))
    ax.set_title(f'{kernel} kernel (acc={acc:.3f})')

plt.tight_layout()
plt.show()
print("RBF handles non-linear data best. Linear fails on moons.")

In [ ]:
# Interview Problem: Explain the kernel trick
print("""
Interview Question: What is the kernel trick in SVM?

Answer:
The kernel trick allows SVM to operate in a high-dimensional,
implicit feature space without actually computing the coordinates
of data in that space. Instead, it computes dot products directly
using a kernel function K(x, y).

This is computationally efficient because:
- Computing φ(x) explicitly would be very expensive
- The kernel function computes φ(x)·φ(y) in O(n) or less
- It enables non-linear decision boundaries with linear algorithms

Common kernels: linear, polynomial, RBF (most popular), sigmoid.
""")

In [ ]:
# Summary
print("=" * 60)
print("KNN & SVM — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. KNN: simple, instance-based, lazy learner — no training phase
2. KNN requires: feature scaling, odd K, appropriate distance metric
3. SVM: finds max-margin hyperplane; uses kernel trick for non-linearity
4. SVM key params: C (regularization), gamma (RBF spread)
5. RBF kernel is the default choice for most non-linear problems
6. Both scale-sensitive: always use StandardScaler or MinMaxScaler
7. SVM works well with clear margins; KNN works well with dense data
""")